In [ ]:
# Run this first to get the correct styling for the Jupyter Notebook
# This cell is nothing to do with the simulation, it's entirely to do with the Jupyter Notebook formatting
from IPython.core.display import HTML
import urllib.request
def css_styling():
    with urllib.request.urlopen("https://raw.githubusercontent.com/uolphysicsteaching/resources/main/notebook.css") as response:
        return HTML(response.read().decode("utf-8"))

try:
    css_styling()
except:
    print("Unable to import formatting data. Likely an internet connectivity issue :( )")

<hr class='double' />

## Full Code Begins

<hr class='double' />

In [ ]:
#
# Module imports
#

# Importing external modules that we'll need
import sys, os
import numpy as np
from matplotlib import pyplot as plt

# Importing the classes we have defined
from modules.SystemParameters import PhysicalConstants
from modules.SystemParameters import SimulationParameters
from modules.PhysicalSystem import PhysicalSystem
from modules.SimulationBox import SimulationBox
from modules.NumericalIntegrator import NumericalIntegrator
from modules.FileIO import FileWriter
from modules.SimulationRunner import SimulationRunner
from modules.SimulationTester import SimulationTester

In [ ]:
#
# Functions defined here (for future use)
#

# Intro stuff
def printIntro():
    
    print("##########################################")
    print("###### Molecular Dynamics Simulator ######")
    print("##########################################")
    print("")

def printSimulationInformation(params):
    print("Simulation Data:\n")
    print("Number of particles: %d" % (params.numParticles))
    print("Volume fraction: %.3f" % (params.volumeFraction))
    print("Temperature: %.2f K" % (params.temperature))
    print("Solvent viscosity: %.2f MPa.ns" % (params.viscosity))
    print("Particle radius: %.2f nm" % (params.particleRadius))
    print("Particle drag: %.2f pN.nm/ns" % (params.drag))
    print("Particle density: %.2f kDa/nm^3" % (params.particleDensity / ((5.0/3.0)*1e-3)))   # Converstion from internal units to user units
    print("Particle mass: %.2f kDa" % (params.particleMass / ((5.0/3.0)*1e-3)))      # Converstion from internal units to user units
    print("kT: %.2f pN.nm" % (params.kbt))
    
    print("")
    print("Timestep: %.2e ns" % (params.dt))
    print("Simulation Time: %.2e ns" % (params.simulationTime))
    print("Frame Rate: %.2e ns" % (params.frameRate))
    print("Number of Steps: %d steps" % (params.numSteps))
    print("Output Rate: %d steps" % (params.outputRate))
    
    print("")
    print("Boundary conditions: %s" % (params.boundaryConditions))
    print("LJ Energy minimum: %.2f pN.nm" % (params.LJeps))
    print("LJ Equilibrium Distance: %.2f nm" % (params.LJr0))
    print("LJ Cutoff Distance: %.2f nm" % (params.LJcutoff))
    
    
    print("")
    print("Equilibration Trajectory Filename: %s" % (params.tFname_Equilibration))
    print("Equilibration Measurement Filename: %s" % (params.mFname_Equilibration))
    print("Production Trajectory Filename: %s" % (params.tFname_Production))
    print("Production Measurement Filename: %s" % (params.mFname_Production))

def printBoxInformation(box):
    print("Simulation Box Data:\n")
    print("Number of voxels: %d (%d,%d,%d)" % (box.totalNumVoxels, box.numVoxels[0], box.numVoxels[1], box.numVoxels[2]))
    print("Dimension: %d,%d,%d" % (box.dimension[0], box.dimension[1], box.dimension[2]))
    print("Voxel Dimension: %d,%d,%d" % (box.resolution, box.resolution, box.resolution))

    print("Boundary Conditions: %s" % (box.bc))

# Outro stuff
def printOutro():
    
    print("##########################################")
    print("######### Thanks for playing! :) #########")
    print("##########################################")
    print("")

---

## Main Program Flow Begins

---

In [ ]:
# Print the simulation information to screen
printIntro()

In [ ]:
# Get some physical constants by instantiating the container class
pConst = PhysicalConstants()

In [ ]:
# Set some parameters (change these to change the physical behaviour of the simulations)
params = SimulationParameters()

# Physical parameters (change these to change the physics of the simulations)
params.simulationType = "Brownian"
params.numParticles = 100   # Number of particles to include in the simulation
params.volumeFraction = 0.01
params.temperature = 150.0   # K (298=room temp)
params.viscosity = 1   # MPa.ns (1 = water)
params.setDensity(1)  # kDa/nm^3 (1 = approximate average protein density, Fischer et al, Protein Sci. 2004 Oct; 13(10): 2825–2828)
                        # There a setter method here for a good reason. Check it out!
    
params.particleRadius = 0.5 # nm

In [ ]:
# Simulation parameters (change these to change the computational behaviour of the simulations)
params.dt = 0.01   # ns (the success and accuracy of your simulation is dependent on this value!)
params.simulationTime = 100 # ns (total amount of simulation time to run)
params.frameRate = 1  # ns (our simulation will output data every "frameRate" frames)

# Box parameters
params.boundaryConditions = ""  # Boundary condition type (hbc:hard boundary conditions)
params.LJeps = 0.0  # Energy minimum for Lennard-Jones interactions
params.LJr0 = 0.0  # Energy minimum for Lennard-Jones interactions
params.LJcutoff = 3.0

params.mFname_Equilibration = ""  # Filename used to write (equilibration) measurement data
params.tFname_Equilibration = ""  # Filename used to write (equilibration) trajectory data
params.mFname_Production = ""  # Filename used to write (production) measurement data
params.tFname_Production = ""  # Filename used to write (production) trajectory data

In [ ]:
# Parameter validation
if not params.validate():
    sys.exit("Inconsistent parameters. Please fix and try again!")

In [ ]:
# Now get all of the "non-primitve" values, calculated from the parameters given
params.calculateNonPrimitives(pConst)

In [ ]:
# Print some more information, so we know we have the right stuff
# If this is wrong, go back and alter the values!
printSimulationInformation(params)

In [ ]:
#
# Build the simulation system
#

# Physical system
try:
    pSys = PhysicalSystem(params.numParticles)
    
except Exception as e:
    print("Error:", e)
    print("Error: Unable to create SimulationBox :(")

In [ ]:
# Computational box
particleVolume = 4.0/3.0 * np.pi * params.particleRadius**3
boxLength = (params.numParticles * particleVolume / params.volumeFraction) ** (1.0/3.0) # Given a volume fraction, how big does the box need to be for N particles?


resolution = params.particleRadius * 10.0 # Should be much bigger than a single particle (+ LJ radius), but nowhere near the size of the box
                                            # There are cleverer ways to calculate this based on the forces involved...see if you can think of one?
# Try to create the box
try:
    box = SimulationBox(params.boundaryConditions, length=boxLength, resolution=resolution)
    
except Exception as e:
    print("Error:", e)
    print("Error: Unable to create SimulationBox :(")

# Info for user
printBoxInformation(box)

In [ ]:
# Build a numerical integrator object
try:
    numericalIntegrator = NumericalIntegrator(params.simulationType)
except Exception as e:
    print("Error: ", e)
    print("Error: Unable to create NumericalIntegrator :(")

In [ ]:
# Get the output files open and ready for writing the equilibration files
# I'll explain what equilibration is shortly!
try:
    fw_Equilibration = FileWriter(params.tFname_Equilibration, params.mFname_Equilibration)
except Exception as e:
    print("Error: ", e)
    print("Error: Unable to create FileWriter for equilibration :(")
    
try:
    fw_Equilibration.openOutputFiles(params)
except Exception as e:
    print("Error: ", e)
    print("Error: Unable to open the equilibration output files for writing :(")

In [ ]:
# Get the output files open and ready for writing the production files
# I'll explain what production is shortly!
try:
    fw_Production = FileWriter(params.tFname_Production, params.mFname_Production)
except Exception as e:
    print("Error: ", e)
    print("Error: Unable to create FileWriter for production :(")
    
try:
    fw_Production.openOutputFiles(params)
except Exception as e:
    print("Error: ", e)
    print("Error: Unable to open the production output files for writing :(")

In [ ]:
#
# Initialise the system
#

# All this will do is randomly distribute the particles throughout the box, ready for initialisation
box.randomInitialise(pSys, params)

In [ ]:
#
# Initialise an object that can perform the actual simulation algorithms
#
runner = SimulationRunner()

In [ ]:
#
# Initialise an object that can test the state of a simulation while it is running
#
tester = SimulationTester()

In [ ]:
#
# Equilibration begins
#

# Add or remove equilibration stages as needed below!

# Think about the equilibration steps you need to perform the simulations you want

In [ ]:
# Calculation for equilibrationTime goes here
equilibrationTime = 1

# Stage 1: Thermal only
runner.runThermalEquilibration(pSys, params, box, fw_Equilibration, equilibrationTime, numericalIntegrator)

In [ ]:
# Test Stage 1 equilibration
tester.testThermalEquilibration(pSys,params)

In [ ]:
#
# Production simulation begins
#

# Calculation for productionTime goes here
productionTime = params.simulationTime
fr = params.frameRate

# Final stage: Production simulation
runner.runProduction(pSys, params, box, fw_Production, productionTime, numericalIntegrator, thermal=True, lj=False, frameRate = fr)

In [ ]:
# Simulation has finished. Clean everything up

# Close simulation files
fw_Equilibration.closeOutputFiles()
fw_Production.closeOutputFiles()

# Delete all objects
del tester
del runner
del fw_Production
del fw_Equilibration
del numericalIntegrator
del box
del pSys
del params

# Print Outro
printOutro()